# Imports and Reproducibility

This notebook implements the CNN replication path for the guitar technique paper using the existing project notebooks as the source of truth for paths, splits, and reporting style.


In [1]:
from pathlib import Path
import copy
import json
import os
import random
import warnings

os.environ.setdefault('NUMBA_CACHE_DIR', str(Path('/tmp') / 'numba_cache_6s955'))
os.environ.setdefault('MPLCONFIGDIR', str(Path('/tmp') / 'mplconfig_6s955'))
os.environ.setdefault('XDG_CACHE_HOME', str(Path('/tmp') / 'xdg_cache_6s955'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    HAS_SEABORN = True
except ModuleNotFoundError:
    sns = None
    HAS_SEABORN = False
try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable if iterable is not None else []
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

warnings.filterwarnings('ignore')
if HAS_SEABORN:
    sns.set_theme(style='whitegrid')

TRAINING_SEED = 0
RANDOM_STATE = 42

def seed_everything(seed: int = TRAINING_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(TRAINING_SEED)

DEVICE = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'mps'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    else 'cpu'
)

print('Torch version:', torch.__version__)
print('Device       :', DEVICE)
print('Training seed:', TRAINING_SEED)
print('Seaborn      :', 'available' if HAS_SEABORN else 'fallback matplotlib mode')


Torch version: 2.11.0
Device       : mps
Training seed: 0
Seaborn      : available


# Project Paths and Configuration

This section intentionally mirrors the existing path and split setup from `processing.ipynb`, while writing CNN outputs to a dedicated stage-2 folder.


In [2]:
PROJECT_ROOT = Path.cwd()

# Fixed dataset path (confirmed working)
DATA_ROOT = PROJECT_ROOT / 'guitar_style_dataset' / 'magcil-guitar_style_dataset-eb27d7b' / 'data'
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Expected DATA_ROOT not found: {DATA_ROOT}')

WAV_ROOT = DATA_ROOT / 'wav'
SPLIT_FILES = {
    'folds': DATA_ROOT / 'folds.json',
    'guitars': DATA_ROOT / 'guitars.json',
    'amplifiers': DATA_ROOT / 'amplifiers.json',
}

CNN_STAGE2_ROOT = PROJECT_ROOT / 'outputs' / 'stage2_cnn_replication'
CNN_RESULTS_ROOT = CNN_STAGE2_ROOT / 'results'
CNN_FIG_ROOT = CNN_STAGE2_ROOT / 'figures'
CNN_CHECKPOINT_ROOT = CNN_STAGE2_ROOT / 'checkpoints'
for p in [CNN_STAGE2_ROOT, CNN_RESULTS_ROOT, CNN_FIG_ROOT, CNN_CHECKPOINT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

EVAL_SPLIT_SCHEMAS = ['folds', 'guitars', 'amplifiers']
DEFAULT_SCHEMA = 'folds'

CLASS_MAPPING = {
    'alternate picking': 0,
    'legato': 1,
    'tapping': 2,
    'sweep picking': 3,
    'vibrato': 4,
    'hammer on': 5,
    'pull off': 6,
    'slide': 7,
    'bend': 8,
}
IDX_TO_CLASS = {v: k for k, v in CLASS_MAPPING.items()}
LABEL_ORDER = sorted(CLASS_MAPPING.values())
CLASS_NAMES_ORDERED = [IDX_TO_CLASS[i] for i in LABEL_ORDER]

TARGET_SR = 8000
SEGMENT_SECONDS = 1.0
SEGMENT_SAMPLES = int(TARGET_SR * SEGMENT_SECONDS)
N_FFT = int(0.05 * TARGET_SR)
WIN_LENGTH = int(0.05 * TARGET_SR)
HOP_LENGTH = int(0.05 * TARGET_SR)
N_MELS = 128
EXPECTED_SEGMENT_SHAPE = (20, 128)

TRAIN_RATIO = 0.8
VALID_RATIO = 0.2
BATCH_SIZE = 16
MAX_EPOCHS = 500
VALIDATE_EVERY = 5
EARLY_STOP_PATIENCE = 10
LEARNING_RATE = 0.002
WEIGHT_DECAY = 0.02
NUM_WORKERS = 0

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT   :', DATA_ROOT)
print('WAV_ROOT    :', WAV_ROOT)
print('Schemas     :', EVAL_SPLIT_SCHEMAS)
print('Output root :', CNN_STAGE2_ROOT)
print('Segment cfg :', f'{SEGMENT_SECONDS}s @ {TARGET_SR} Hz -> {EXPECTED_SEGMENT_SHAPE}')


PROJECT_ROOT: /Users/juansantiago/Desktop/MIT Classes 2023-2027/Spring 2026/6.S955/FinalProject/6.S955_project
DATA_ROOT   : /Users/juansantiago/Desktop/MIT Classes 2023-2027/Spring 2026/6.S955/FinalProject/6.S955_project/guitar_style_dataset/magcil-guitar_style_dataset-eb27d7b/data
WAV_ROOT    : /Users/juansantiago/Desktop/MIT Classes 2023-2027/Spring 2026/6.S955/FinalProject/6.S955_project/guitar_style_dataset/magcil-guitar_style_dataset-eb27d7b/data/wav
Schemas     : ['folds', 'guitars', 'amplifiers']
Output root : /Users/juansantiago/Desktop/MIT Classes 2023-2027/Spring 2026/6.S955/FinalProject/6.S955_project/outputs/stage2_cnn_replication
Segment cfg : 1.0s @ 8000 Hz -> (20, 128)


# Class Mapping and Split Loading

The file registry is built from the dataset folders, and the official JSON splits are loaded in the same file-level style as the baseline notebook.


In [3]:
def build_file_registry(wav_root: Path, class_mapping: dict[str, int]) -> pd.DataFrame:
    rows = []
    class_dirs = [p for p in sorted(wav_root.iterdir()) if p.is_dir()]
    for class_dir in class_dirs:
        class_name = class_dir.name
        if class_name not in class_mapping:
            raise KeyError(f'Unexpected class folder: {class_name}')
        label = class_mapping[class_name]
        for wav_path in sorted(class_dir.glob('*.wav')):
            rows.append(
                {
                    'file_name': wav_path.name,
                    'file_path': str(wav_path),
                    'class_name': class_name,
                    'label': label,
                }
            )
    registry_df = pd.DataFrame(rows).sort_values(['label', 'file_name']).reset_index(drop=True)
    if registry_df.empty:
        raise ValueError('No WAV files were found in the dataset root.')
    return registry_df


def load_split_dataframe(schema: str, registry_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    split_obj = json.loads(SPLIT_FILES[schema].read_text())
    rows = []
    for fold_name, data in split_obj.items():
        for partition in ['train', 'test']:
            for file_name in data.get(partition, []):
                rows.append(
                    {
                        'schema': schema,
                        'fold': str(fold_name),
                        'partition': partition,
                        'file_name': Path(str(file_name)).name,
                    }
                )
    split_df = pd.DataFrame(rows).drop_duplicates()
    model_df = split_df.merge(registry_df, on='file_name', how='inner')
    missing = sorted(set(split_df['file_name']) - set(model_df['file_name']))
    if missing:
        raise ValueError(f'Missing files after split join for {schema}: {missing[:10]}')
    return split_df, model_df


FILE_REGISTRY_DF = build_file_registry(WAV_ROOT, CLASS_MAPPING)

SPLIT_TABLES = {}
MODEL_TABLES = {}
EXPECTED_SPLIT_COUNTS = {
    'folds': {'train': 441, 'test': 108, 'num_folds': 5},
    'guitars': {'train': 366, 'test': 183, 'num_folds': 3},
    'amplifiers': {'train': 366, 'test': 183, 'num_folds': 3},
}

for schema in EVAL_SPLIT_SCHEMAS:
    split_df, model_df = load_split_dataframe(schema, FILE_REGISTRY_DF)
    SPLIT_TABLES[schema] = split_df
    MODEL_TABLES[schema] = model_df

    fold_counts = model_df.groupby(['fold', 'partition'])['file_name'].nunique().unstack(fill_value=0)
    expected = EXPECTED_SPLIT_COUNTS[schema]
    assert len(fold_counts) == expected['num_folds'], f'Unexpected number of folds for {schema}'
    assert fold_counts['train'].eq(expected['train']).all(), f'Unexpected train counts for {schema}'
    assert fold_counts['test'].eq(expected['test']).all(), f'Unexpected test counts for {schema}'

    print(f'[{schema}] folds={list(fold_counts.index)}')
    print(fold_counts)
    print()

FILE_REGISTRY_DF.head()


[folds] folds=['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']
partition  test  train
fold                  
fold_0      108    441
fold_1      108    441
fold_2      108    441
fold_3      108    441
fold_4      108    441

[guitars] folds=['carvin', 'palm-bay', 'rwg']
partition  test  train
fold                  
carvin      183    366
palm-bay    183    366
rwg         183    366

[amplifiers] folds=['dc-modern', 'plexiglass', 'sl100']
partition   test  train
fold                   
dc-modern    183    366
plexiglass   183    366
sl100        183    366



,file_name,file_path,class_name,label
0,class_0_carvin_dc-modern_1.wav,/Users/juansantiago/Desktop/MIT Classes 2023-2...,alternate picking,0
1,class_0_carvin_dc-modern_1_1.wav,/Users/juansantiago/Desktop/MIT Classes 2023-2...,alternate picking,0
2,class_0_carvin_dc-modern_1_2.wav,/Users/juansantiago/Desktop/MIT Classes 2023-2...,alternate picking,0
3,class_0_carvin_dc-modern_2.wav,/Users/juansantiago/Desktop/MIT Classes 2023-2...,alternate picking,0
4,class_0_carvin_dc-modern_2_1.wav,/Users/juansantiago/Desktop/MIT Classes 2023-2...,alternate picking,0


# Audio Segmentation and Mel-Spectrogram Extraction

Features are computed in memory to avoid writing segmented audio back to disk. Each WAV file is trimmed to full seconds, split into non-overlapping 1-second chunks, and converted to log-mel features with the agreed `20 x 128` geometry.


In [4]:
def load_audio_mono_8k(file_path: str, target_sr: int = TARGET_SR) -> tuple[np.ndarray, int]:
    waveform, sample_rate = librosa.load(file_path, sr=target_sr, mono=True)
    return waveform.astype(np.float32), sample_rate


def trim_to_integer_seconds(
    waveform: np.ndarray,
    sample_rate: int = TARGET_SR,
    segment_seconds: float = SEGMENT_SECONDS,
) -> np.ndarray:
    segment_samples = int(sample_rate * segment_seconds)
    usable_samples = (len(waveform) // segment_samples) * segment_samples
    return waveform[:usable_samples]


def segment_waveform(
    waveform: np.ndarray,
    sample_rate: int = TARGET_SR,
    segment_seconds: float = SEGMENT_SECONDS,
) -> np.ndarray:
    segment_samples = int(sample_rate * segment_seconds)
    trimmed = trim_to_integer_seconds(waveform, sample_rate, segment_seconds)
    if trimmed.size == 0:
        return np.empty((0, segment_samples), dtype=np.float32)
    return trimmed.reshape(-1, segment_samples).astype(np.float32)


def compute_log_mel_segment(segment: np.ndarray, sample_rate: int = TARGET_SR) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=segment,
        sr=sample_rate,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        n_mels=N_MELS,
        center=False,
        power=2.0,
    )
    log_mel = librosa.power_to_db(mel, ref=np.max).T.astype(np.float32)
    if log_mel.shape != EXPECTED_SEGMENT_SHAPE:
        raise ValueError(f'Expected {EXPECTED_SEGMENT_SHAPE}, got {log_mel.shape}')
    return log_mel


def extract_file_segment_features(file_path: str) -> np.ndarray:
    waveform, sample_rate = load_audio_mono_8k(file_path)
    segments = segment_waveform(waveform, sample_rate, SEGMENT_SECONDS)
    if segments.shape[0] == 0:
        return np.empty((0, *EXPECTED_SEGMENT_SHAPE), dtype=np.float32)
    features = np.stack([compute_log_mel_segment(segment, sample_rate) for segment in segments], axis=0)
    return features.astype(np.float32)


def build_feature_cache(file_registry_df: pd.DataFrame) -> tuple[dict[str, np.ndarray], pd.DataFrame]:
    cache = {}
    rows = []
    iterator = tqdm(
        file_registry_df.itertuples(index=False),
        total=len(file_registry_df),
        desc='Caching file-level log-mel features',
    )
    for row in iterator:
        features = extract_file_segment_features(row.file_path)
        if features.shape[0] == 0:
            raise ValueError(f'File produced zero full segments: {row.file_name}')
        cache[row.file_name] = features
        rows.append({'file_name': row.file_name, 'num_segments': int(features.shape[0])})
    return cache, pd.DataFrame(rows)


FILE_SEGMENT_CACHE, FILE_SEGMENT_COUNTS = build_feature_cache(FILE_REGISTRY_DF)
FILE_REGISTRY_DF = FILE_REGISTRY_DF.merge(FILE_SEGMENT_COUNTS, on='file_name', how='left')
for schema in EVAL_SPLIT_SCHEMAS:
    MODEL_TABLES[schema] = MODEL_TABLES[schema].drop(columns=['num_segments'], errors='ignore').merge(
        FILE_SEGMENT_COUNTS, on='file_name', how='left'
    )

sample_row = FILE_REGISTRY_DF.iloc[0]
sample_features = FILE_SEGMENT_CACHE[sample_row['file_name']]
assert sample_features.shape[1:] == EXPECTED_SEGMENT_SHAPE

print('Cached files :', len(FILE_SEGMENT_CACHE))
print('Sample file  :', sample_row['file_name'])
print('Sample tensor:', sample_features.shape)
print('Total segments cached:', int(FILE_SEGMENT_COUNTS['num_segments'].sum()))
FILE_SEGMENT_COUNTS.describe()


Caching file-level log-mel features:   0%|          | 0/549 [00:00<?, ?it/s]

Cached files : 549
Sample file  : class_0_carvin_dc-modern_1.wav
Sample tensor: (5, 20, 128)
Total segments cached: 6044


,num_segments
count,549.000000
mean,11.009107
std,9.807728
min,2.000000
25%,3.000000
50%,8.000000
75%,20.000000
max,41.000000


# PyTorch Dataset and DataLoader

Training uses segment-level tensors derived from the train files only. The internal train/validation split is stratified at the segment level with seed `0`, matching the upstream CNN workflow.


In [5]:
class SegmentTensorDataset(Dataset):
    def __init__(self, features: np.ndarray, labels: np.ndarray):
        self.features = torch.from_numpy(features.astype(np.float32))
        self.labels = torch.from_numpy(labels.astype(np.int64))

    def __len__(self) -> int:
        return int(self.labels.shape[0])

    def __getitem__(self, index: int):
        return self.features[index], self.labels[index]


def build_segment_index(file_df: pd.DataFrame, feature_cache: dict[str, np.ndarray]) -> pd.DataFrame:
    rows = []
    for row in file_df.itertuples(index=False):
        num_segments = int(feature_cache[row.file_name].shape[0])
        for segment_idx in range(num_segments):
            rows.append(
                {
                    'file_name': row.file_name,
                    'file_path': row.file_path,
                    'class_name': row.class_name,
                    'label': int(row.label),
                    'segment_idx': int(segment_idx),
                }
            )
    return pd.DataFrame(rows)


def materialize_segment_arrays(
    segment_df: pd.DataFrame,
    feature_cache: dict[str, np.ndarray],
) -> tuple[np.ndarray, np.ndarray]:
    features = np.stack(
        [feature_cache[row.file_name][row.segment_idx] for row in segment_df.itertuples(index=False)], axis=0
    ).astype(np.float32)
    labels = segment_df['label'].to_numpy(dtype=np.int64)
    return features, labels


def build_train_val_dataloaders(
    train_file_df: pd.DataFrame,
    feature_cache: dict[str, np.ndarray],
    batch_size: int = BATCH_SIZE,
    valid_ratio: float = VALID_RATIO,
    seed: int = TRAINING_SEED,
) -> tuple[DataLoader, DataLoader, dict]:
    segment_df = build_segment_index(train_file_df, feature_cache)
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=TRAIN_RATIO,
        test_size=valid_ratio,
        random_state=seed,
    )
    train_idx, valid_idx = next(splitter.split(np.zeros(len(segment_df)), segment_df['label']))

    train_segment_df = segment_df.iloc[train_idx].reset_index(drop=True)
    valid_segment_df = segment_df.iloc[valid_idx].reset_index(drop=True)

    x_train, y_train = materialize_segment_arrays(train_segment_df, feature_cache)
    x_valid, y_valid = materialize_segment_arrays(valid_segment_df, feature_cache)

    train_dataset = SegmentTensorDataset(x_train, y_train)
    valid_dataset = SegmentTensorDataset(x_valid, y_valid)

    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=NUM_WORKERS,
        generator=generator,
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
    )

    split_meta = {
        'n_train_files': int(train_file_df['file_name'].nunique()),
        'n_train_segments': int(len(train_segment_df)),
        'n_valid_segments': int(len(valid_segment_df)),
        'train_segment_df': train_segment_df,
        'valid_segment_df': valid_segment_df,
    }
    return train_loader, valid_loader, split_meta


demo_fold = sorted(MODEL_TABLES[DEFAULT_SCHEMA]['fold'].unique())[0]
demo_train_file_df = MODEL_TABLES[DEFAULT_SCHEMA].query("fold == @demo_fold and partition == 'train'").reset_index(drop=True)
demo_train_loader, demo_valid_loader, demo_split_meta = build_train_val_dataloaders(
    demo_train_file_df,
    FILE_SEGMENT_CACHE,
)
demo_batch_x, demo_batch_y = next(iter(demo_train_loader))

assert tuple(demo_batch_x.shape[1:]) == EXPECTED_SEGMENT_SHAPE

print('Demo fold        :', demo_fold)
print('Train batch shape:', tuple(demo_batch_x.shape))
print('Label batch shape:', tuple(demo_batch_y.shape))
print('Split metadata   :', {k: v for k, v in demo_split_meta.items() if not k.endswith('_df')})


Demo fold        : fold_0
Train batch shape: (16, 20, 128)
Label batch shape: (16,)
Split metadata   : {'n_train_files': 441, 'n_train_segments': 4003, 'n_valid_segments': 1001}


# CNN Model Definition (Paper Architecture)

The convolutional stack follows the paper contract, and the classifier head uses the agreed `2048 -> 1048 -> 256 -> 9` mapping with the upstream dropout pattern.


In [6]:
class GuitarTechniqueCNN(nn.Module):
    def __init__(self, num_classes: int = len(CLASS_MAPPING)):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(64, 128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(128, 256, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.75),
            nn.Linear(2048, 1048),
            nn.LeakyReLU(),
            nn.Dropout(0.5),
            nn.Linear(1048, 256),
            nn.LeakyReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim == 3:
            x = x.unsqueeze(1)
        x = self.feature_extractor(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)


model = GuitarTechniqueCNN(num_classes=len(CLASS_MAPPING))
dummy_logits = model(torch.randn(4, *EXPECTED_SEGMENT_SHAPE))

assert tuple(dummy_logits.shape) == (4, len(CLASS_MAPPING))
assert model.classifier[1].in_features == 2048

print(model)
print('Flattened feature size:', model.classifier[1].in_features)


GuitarTechniqueCNN(
  (feature_extractor): Sequential(
    (0): Conv2d(1, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.01)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): LeakyReLU(negative_slope=0.01)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): LeakyReLU(negative_slope=0.01)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))


# Training Loop

Training follows the resolved upstream defaults: AdamW, cross-entropy, a 500-epoch budget, validation every 5 epochs, and early stopping based on validation macro-F1.


In [7]:
def summarize_classification(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    }


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device = DEVICE,
) -> dict[str, float]:
    model.train()
    losses = []
    y_true = []
    y_pred = []

    for features, labels in dataloader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        losses.append(float(loss.item()))
        y_true.append(labels.detach().cpu().numpy())
        y_pred.append(logits.detach().cpu().argmax(dim=1).numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    metrics = summarize_classification(y_true, y_pred)
    metrics['loss'] = float(np.mean(losses))
    return metrics


@torch.no_grad()
def evaluate_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device = DEVICE,
) -> dict[str, float]:
    model.eval()
    losses = []
    y_true = []
    y_pred = []

    for features, labels in dataloader:
        features = features.to(device)
        labels = labels.to(device)
        logits = model(features)
        loss = criterion(logits, labels)

        losses.append(float(loss.item()))
        y_true.append(labels.detach().cpu().numpy())
        y_pred.append(logits.detach().cpu().argmax(dim=1).numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    metrics = summarize_classification(y_true, y_pred)
    metrics['loss'] = float(np.mean(losses))
    return metrics


def fit_fold_model(
    schema: str,
    fold_name: str,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    device: torch.device = DEVICE,
) -> tuple[nn.Module, pd.DataFrame, dict]:
    seed_everything(TRAINING_SEED)
    model = GuitarTechniqueCNN(num_classes=len(CLASS_MAPPING)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    history_rows = []
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    best_valid_f1 = -np.inf
    best_valid_accuracy = 0.0
    checks_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device=device)
        row = {
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'train_accuracy': train_metrics['accuracy'],
            'train_f1_macro': train_metrics['f1_macro'],
        }

        if epoch % VALIDATE_EVERY == 0 or epoch == MAX_EPOCHS:
            valid_metrics = evaluate_one_epoch(model, valid_loader, criterion, device=device)
            row.update(
                {
                    'valid_loss': valid_metrics['loss'],
                    'valid_accuracy': valid_metrics['accuracy'],
                    'valid_f1_macro': valid_metrics['f1_macro'],
                }
            )
            improved = valid_metrics['f1_macro'] > best_valid_f1 + 1e-6
            if improved:
                best_valid_f1 = valid_metrics['f1_macro']
                best_valid_accuracy = valid_metrics['accuracy']
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())
                checks_without_improvement = 0
            else:
                checks_without_improvement += 1

            print(
                f'[{schema} | {fold_name}] epoch={epoch:03d} '
                f'train_loss={train_metrics["loss"]:.4f} '
                f'valid_f1={valid_metrics["f1_macro"]:.4f} '
                f'valid_acc={valid_metrics["accuracy"]:.4f}'
            )

            history_rows.append(row)

            if checks_without_improvement >= EARLY_STOP_PATIENCE:
                print(f'[{schema} | {fold_name}] early stopping at epoch {epoch}. Best epoch: {best_epoch}')
                break
        else:
            history_rows.append(row)

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history_rows)
    fit_summary = {
        'best_epoch': int(best_epoch),
        'best_valid_f1_macro': float(best_valid_f1),
        'best_valid_accuracy': float(best_valid_accuracy),
    }
    return model, history_df, fit_summary


# Fold Evaluation and Aggregated Metrics

Each held-out file is scored by segment, then collapsed back to a single file prediction by majority vote with mean-posterior tie breaking.


In [8]:
def majority_vote_with_tiebreak(segment_probabilities: np.ndarray) -> tuple[int, np.ndarray]:
    segment_predictions = segment_probabilities.argmax(axis=1)
    vote_counts = np.bincount(segment_predictions, minlength=len(LABEL_ORDER))
    top_vote_count = vote_counts.max()
    tied_labels = np.flatnonzero(vote_counts == top_vote_count)
    if len(tied_labels) == 1:
        return int(tied_labels[0]), vote_counts
    mean_probabilities = segment_probabilities.mean(axis=0)
    best_tied_index = tied_labels[np.argmax(mean_probabilities[tied_labels])]
    return int(best_tied_index), vote_counts


@torch.no_grad()
def predict_file_majority_vote(
    model: nn.Module,
    file_name: str,
    feature_cache: dict[str, np.ndarray],
    device: torch.device = DEVICE,
) -> dict:
    file_features = feature_cache[file_name]
    feature_tensor = torch.from_numpy(file_features).to(device)
    logits = model(feature_tensor)
    probabilities = torch.softmax(logits, dim=1).detach().cpu().numpy()
    pred_label, vote_counts = majority_vote_with_tiebreak(probabilities)
    mean_probabilities = probabilities.mean(axis=0)
    return {
        'pred_label': int(pred_label),
        'pred_class': IDX_TO_CLASS[int(pred_label)],
        'num_segments': int(file_features.shape[0]),
        'vote_count': int(vote_counts[pred_label]),
        'mean_confidence': float(mean_probabilities[pred_label]),
    }


def save_fold_checkpoint(
    schema: str,
    fold_name: str,
    model: nn.Module,
    history_df: pd.DataFrame,
    fit_summary: dict,
    split_meta: dict,
) -> Path:
    checkpoint_path = CNN_CHECKPOINT_ROOT / f'{schema}_{fold_name}_best.pt'
    payload = {
        'schema': schema,
        'fold': fold_name,
        'model_state_dict': model.state_dict(),
        'class_mapping': CLASS_MAPPING,
        'segment_shape': EXPECTED_SEGMENT_SHAPE,
        'sample_rate': TARGET_SR,
        'segment_seconds': SEGMENT_SECONDS,
        'n_fft': N_FFT,
        'win_length': WIN_LENGTH,
        'hop_length': HOP_LENGTH,
        'n_mels': N_MELS,
        'history': history_df.to_dict(orient='records'),
        'fit_summary': fit_summary,
        'split_meta': {k: v for k, v in split_meta.items() if not k.endswith('_df')},
    }
    torch.save(payload, checkpoint_path)
    return checkpoint_path


def run_single_fold(schema: str, fold_name: str, schema_df: pd.DataFrame) -> dict:
    fold_df = schema_df[schema_df['fold'] == fold_name].copy().reset_index(drop=True)
    train_file_df = fold_df[fold_df['partition'] == 'train'].copy().reset_index(drop=True)
    test_file_df = fold_df[fold_df['partition'] == 'test'].copy().reset_index(drop=True)

    train_loader, valid_loader, split_meta = build_train_val_dataloaders(train_file_df, FILE_SEGMENT_CACHE)
    model, history_df, fit_summary = fit_fold_model(schema, fold_name, train_loader, valid_loader, device=DEVICE)
    model.eval()

    prediction_rows = []
    for row in tqdm(
        test_file_df.itertuples(index=False),
        total=len(test_file_df),
        desc=f'Testing {schema} {fold_name}',
        leave=False,
    ):
        pred_info = predict_file_majority_vote(model, row.file_name, FILE_SEGMENT_CACHE, device=DEVICE)
        prediction_rows.append(
            {
                'fold': fold_name,
                'file_name': row.file_name,
                'class_name': row.class_name,
                'label': int(row.label),
                'pred_label': pred_info['pred_label'],
                'pred_class': pred_info['pred_class'],
                'num_segments': pred_info['num_segments'],
                'vote_count': pred_info['vote_count'],
                'mean_confidence': pred_info['mean_confidence'],
            }
        )

    pred_df = pd.DataFrame(prediction_rows)
    y_true = pred_df['label'].to_numpy(dtype=np.int64)
    y_pred = pred_df['pred_label'].to_numpy(dtype=np.int64)
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    metrics = summarize_classification(y_true, y_pred)
    report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=LABEL_ORDER,
            target_names=CLASS_NAMES_ORDERED,
            output_dict=True,
            zero_division=0,
        )
    ).T

    checkpoint_path = save_fold_checkpoint(schema, fold_name, model, history_df, fit_summary, split_meta)
    fold_metrics = {
        'fold': fold_name,
        'n_train_files': int(train_file_df['file_name'].nunique()),
        'n_test_files': int(test_file_df['file_name'].nunique()),
        'n_train_segments': int(split_meta['n_train_segments']),
        'n_valid_segments': int(split_meta['n_valid_segments']),
        'best_epoch': int(fit_summary['best_epoch']),
        'best_valid_accuracy': float(fit_summary['best_valid_accuracy']),
        'best_valid_f1_macro': float(fit_summary['best_valid_f1_macro']),
        'test_accuracy': float(metrics['accuracy']),
        'test_f1_macro': float(metrics['f1_macro']),
        'checkpoint_path': str(checkpoint_path),
    }

    return {
        'fold_metrics': fold_metrics,
        'pred_df': pred_df,
        'cm': cm,
        'report_df': report_df,
        'history_df': history_df,
    }


def run_schema_experiment(schema: str) -> dict:
    schema_df = MODEL_TABLES[schema].copy()
    folds = sorted(schema_df['fold'].unique().tolist())
    aggregated_cm = np.zeros((len(LABEL_ORDER), len(LABEL_ORDER)), dtype=int)
    all_fold_metrics = []
    all_pred_rows = []
    fold_cms = {}

    for fold_name in folds:
        print(f'\nRunning {schema} / {fold_name}')
        fold_result = run_single_fold(schema, fold_name, schema_df)
        all_fold_metrics.append(fold_result['fold_metrics'])
        all_pred_rows.append(fold_result['pred_df'])
        fold_cms[fold_name] = fold_result['cm']
        aggregated_cm += fold_result['cm']
        fold_result['report_df'].to_csv(CNN_RESULTS_ROOT / f'{schema}_{fold_name}_classification_report.csv')

    fold_metrics_df = pd.DataFrame(all_fold_metrics)
    pred_all_df = pd.concat(all_pred_rows, ignore_index=True)

    fold_metrics_df.to_csv(CNN_RESULTS_ROOT / f'{schema}_fold_metrics.csv', index=False)
    pred_all_df.to_csv(CNN_RESULTS_ROOT / f'{schema}_file_predictions.csv', index=False)
    np.save(CNN_RESULTS_ROOT / f'{schema}_aggregated_confusion_matrix.npy', aggregated_cm)

    summary = {
        'schema': schema,
        'num_folds': int(len(folds)),
        'accuracy_mean': float(fold_metrics_df['test_accuracy'].mean()),
        'accuracy_std': float(fold_metrics_df['test_accuracy'].std(ddof=0)),
        'f1_macro_mean': float(fold_metrics_df['test_f1_macro'].mean()),
        'f1_macro_std': float(fold_metrics_df['test_f1_macro'].std(ddof=0)),
        'cnn_replication': True,
    }
    (CNN_RESULTS_ROOT / f'{schema}_summary.json').write_text(json.dumps(summary, indent=2))

    return {
        'schema': schema,
        'folds': folds,
        'fold_metrics_df': fold_metrics_df,
        'pred_all_df': pred_all_df,
        'fold_cms': fold_cms,
        'agg_cm': aggregated_cm,
        'summary': summary,
    }


In [9]:
RUN_ALL_EXPERIMENTS = True
SCHEMAS_TO_RUN = list(EVAL_SPLIT_SCHEMAS)

experiment_results = {}
if RUN_ALL_EXPERIMENTS:
    for schema in SCHEMAS_TO_RUN:
        print(f'\n' + '=' * 80)
        print(f'Executing schema: {schema}')
        print('=' * 80)
        experiment_results[schema] = run_schema_experiment(schema)
else:
    print('Set RUN_ALL_EXPERIMENTS = True to execute the full paper-style evaluation.')

list(experiment_results.keys())



Executing schema: folds

Running folds / fold_0
[folds | fold_0] epoch=005 train_loss=0.7529 valid_f1=0.5026 valid_acc=0.5045
[folds | fold_0] epoch=010 train_loss=0.5074 valid_f1=0.6732 valid_acc=0.6883
[folds | fold_0] epoch=015 train_loss=0.4119 valid_f1=0.3273 valid_acc=0.4436
[folds | fold_0] epoch=020 train_loss=0.2805 valid_f1=0.6861 valid_acc=0.7273
[folds | fold_0] epoch=025 train_loss=0.1884 valid_f1=0.8440 valid_acc=0.8362
[folds | fold_0] epoch=030 train_loss=0.1667 valid_f1=0.8890 valid_acc=0.8911
[folds | fold_0] epoch=035 train_loss=0.1148 valid_f1=0.9244 valid_acc=0.9191
[folds | fold_0] epoch=040 train_loss=0.2603 valid_f1=0.8623 valid_acc=0.8831
[folds | fold_0] epoch=045 train_loss=0.1312 valid_f1=0.9133 valid_acc=0.9031
[folds | fold_0] epoch=050 train_loss=0.1370 valid_f1=0.9101 valid_acc=0.9101
[folds | fold_0] epoch=055 train_loss=0.1223 valid_f1=0.9092 valid_acc=0.9091
[folds | fold_0] epoch=060 train_loss=0.0534 valid_f1=0.9237 valid_acc=0.9211
[folds | fold_0

Testing folds fold_0:   0%|          | 0/108 [00:00<?, ?it/s]


Running folds / fold_1
[folds | fold_1] epoch=005 train_loss=0.7145 valid_f1=0.6095 valid_acc=0.6471
[folds | fold_1] epoch=010 train_loss=0.4903 valid_f1=0.5341 valid_acc=0.4115
[folds | fold_1] epoch=015 train_loss=0.3328 valid_f1=0.8890 valid_acc=0.8796
[folds | fold_1] epoch=020 train_loss=0.2912 valid_f1=0.8608 valid_acc=0.8344
[folds | fold_1] epoch=025 train_loss=0.1883 valid_f1=0.8626 valid_acc=0.8508
[folds | fold_1] epoch=030 train_loss=0.1480 valid_f1=0.6175 valid_acc=0.6698
[folds | fold_1] epoch=035 train_loss=0.1084 valid_f1=0.8993 valid_acc=0.8755
[folds | fold_1] epoch=040 train_loss=0.0833 valid_f1=0.9102 valid_acc=0.8940
[folds | fold_1] epoch=045 train_loss=0.1166 valid_f1=0.8734 valid_acc=0.8313
[folds | fold_1] epoch=050 train_loss=0.1137 valid_f1=0.9244 valid_acc=0.9146
[folds | fold_1] epoch=055 train_loss=0.1146 valid_f1=0.9194 valid_acc=0.9146
[folds | fold_1] epoch=060 train_loss=0.0756 valid_f1=0.9343 valid_acc=0.9198
[folds | fold_1] epoch=065 train_loss=0.

Testing folds fold_1:   0%|          | 0/108 [00:00<?, ?it/s]


Running folds / fold_2
[folds | fold_2] epoch=005 train_loss=0.6996 valid_f1=0.4984 valid_acc=0.5510
[folds | fold_2] epoch=010 train_loss=0.4371 valid_f1=0.8099 valid_acc=0.8133
[folds | fold_2] epoch=015 train_loss=0.2855 valid_f1=0.8668 valid_acc=0.8385
[folds | fold_2] epoch=020 train_loss=0.2580 valid_f1=0.8938 valid_acc=0.8880
[folds | fold_2] epoch=025 train_loss=0.1489 valid_f1=0.9231 valid_acc=0.9051
[folds | fold_2] epoch=030 train_loss=0.1346 valid_f1=0.8910 valid_acc=0.8890
[folds | fold_2] epoch=035 train_loss=0.0948 valid_f1=0.8905 valid_acc=0.8860
[folds | fold_2] epoch=040 train_loss=0.1034 valid_f1=0.9356 valid_acc=0.9102
[folds | fold_2] epoch=045 train_loss=0.1074 valid_f1=0.8890 valid_acc=0.8840
[folds | fold_2] epoch=050 train_loss=0.2023 valid_f1=0.9290 valid_acc=0.9142
[folds | fold_2] epoch=055 train_loss=0.0713 valid_f1=0.9220 valid_acc=0.9092
[folds | fold_2] epoch=060 train_loss=0.0898 valid_f1=0.7892 valid_acc=0.7669
[folds | fold_2] epoch=065 train_loss=0.

...


Running amplifiers / plexiglass
[amplifiers | plexiglass] epoch=005 train_loss=0.7927 valid_f1=0.5159 valid_acc=0.5159
[amplifiers | plexiglass] epoch=010 train_loss=0.5516 valid_f1=0.6980 valid_acc=0.6789
[amplifiers | plexiglass] epoch=015 train_loss=0.4218 valid_f1=0.8049 valid_acc=0.7574
[amplifiers | plexiglass] epoch=020 train_loss=0.3092 valid_f1=0.7168 valid_acc=0.7463
[amplifiers | plexiglass] epoch=025 train_loss=0.2185 valid_f1=0.5600 valid_acc=0.6005
[amplifiers | plexiglass] epoch=030 train_loss=0.1357 valid_f1=0.8986 valid_acc=0.8909
[amplifiers | plexiglass] epoch=035 train_loss=0.1480 valid_f1=0.9232 valid_acc=0.9044
[amplifiers | plexiglass] epoch=040 train_loss=0.1528 valid_f1=0.8904 valid_acc=0.8836
[amplifiers | plexiglass] epoch=045 train_loss=0.0890 valid_f1=0.8046 valid_acc=0.7831
[amplifiers | plexiglass] epoch=050 train_loss=0.1064 valid_f1=0.8368 valid_acc=0.8150
[amplifiers | plexiglass] epoch=055 train_loss=0.0982 valid_f1=0.9169 valid_acc=0.8897
[amplifier

Testing amplifiers plexiglass:   0%|          | 0/183 [00:00<?, ?it/s]


Running amplifiers / sl100
[amplifiers | sl100] epoch=005 train_loss=0.7565 valid_f1=0.6111 valid_acc=0.6419
[amplifiers | sl100] epoch=010 train_loss=0.5483 valid_f1=0.5917 valid_acc=0.5844
[amplifiers | sl100] epoch=015 train_loss=0.4465 valid_f1=0.7844 valid_acc=0.7442
[amplifiers | sl100] epoch=020 train_loss=0.3404 valid_f1=0.6105 valid_acc=0.6701
[amplifiers | sl100] epoch=025 train_loss=0.2410 valid_f1=0.8118 valid_acc=0.7980
[amplifiers | sl100] epoch=030 train_loss=0.1775 valid_f1=0.9169 valid_acc=0.9003
[amplifiers | sl100] epoch=035 train_loss=0.1040 valid_f1=0.9139 valid_acc=0.9284
[amplifiers | sl100] epoch=040 train_loss=0.0996 valid_f1=0.9327 valid_acc=0.9348
[amplifiers | sl100] epoch=045 train_loss=0.0862 valid_f1=0.9448 valid_acc=0.9412
[amplifiers | sl100] epoch=050 train_loss=0.0783 valid_f1=0.9298 valid_acc=0.9373
[amplifiers | sl100] epoch=055 train_loss=0.0588 valid_f1=0.9494 valid_acc=0.9527
[amplifiers | sl100] epoch=060 train_loss=0.1122 valid_f1=0.6303 valid

Testing amplifiers sl100:   0%|          | 0/183 [00:00<?, ?it/s]

['folds', 'guitars', 'amplifiers']

# Confusion Matrices and Plots

Confusion matrices are saved per fold and per schema using the same general visual style as the baseline notebook.


In [10]:
def save_confusion_matrix_plot(cm: np.ndarray, title: str, output_path: Path) -> None:
    plt.figure(figsize=(8, 6))
    if HAS_SEABORN:
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=CLASS_NAMES_ORDERED,
            yticklabels=CLASS_NAMES_ORDERED,
        )
    else:
        plt.imshow(cm, cmap='Blues')
        plt.colorbar()
        for row_idx in range(cm.shape[0]):
            for col_idx in range(cm.shape[1]):
                plt.text(col_idx, row_idx, int(cm[row_idx, col_idx]), ha='center', va='center', fontsize=8)
        plt.xticks(range(len(CLASS_NAMES_ORDERED)), CLASS_NAMES_ORDERED, rotation=45, ha='right')
        plt.yticks(range(len(CLASS_NAMES_ORDERED)), CLASS_NAMES_ORDERED)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(output_path, dpi=220)
    plt.close()


if not experiment_results:
    print('No experiment results found yet. Run the evaluation cell in Section 8 first.')
else:
    for schema, result in experiment_results.items():
        for fold_name, cm in result['fold_cms'].items():
            save_confusion_matrix_plot(
                cm,
                f'Confusion Matrix - {schema} {fold_name}',
                CNN_FIG_ROOT / f'cm_{schema}_{fold_name}.png',
            )
        save_confusion_matrix_plot(
            result['agg_cm'],
            f'Aggregated Confusion Matrix - {schema}',
            CNN_FIG_ROOT / f'cm_{schema}_aggregated.png',
        )
    print('Saved confusion matrices to:', CNN_FIG_ROOT)


Saved confusion matrices to: /Users/juansantiago/Desktop/MIT Classes 2023-2027/Spring 2026/6.S955/FinalProject/6.S955_project/outputs/stage2_cnn_replication/figures


# Final Result Summary (Paper-Style Table)

This cell assembles the three experiment summaries into a single paper-style table and saves it for reference.


In [11]:
experiment_labels = {
    'folds': 'Experiment 1',
    'guitars': 'Experiment 2',
    'amplifiers': 'Experiment 3',
}

summary_rows = []
for schema in EVAL_SPLIT_SCHEMAS:
    if schema not in experiment_results:
        continue
    summary = experiment_results[schema]['summary']
    summary_rows.append(
        {
            'Experiment': experiment_labels[schema],
            'Split': schema,
            'Method': 'CNN',
            'Accuracy Mean (%)': 100.0 * summary['accuracy_mean'],
            'Accuracy Std (%)': 100.0 * summary['accuracy_std'],
            'Macro F1 Mean (%)': 100.0 * summary['f1_macro_mean'],
            'Macro F1 Std (%)': 100.0 * summary['f1_macro_std'],
        }
    )

paper_style_summary_df = pd.DataFrame(summary_rows)
if not paper_style_summary_df.empty:
    paper_style_summary_df.to_csv(CNN_RESULTS_ROOT / 'paper_style_summary.csv', index=False)
    print('Saved:', CNN_RESULTS_ROOT / 'paper_style_summary.csv')
else:
    print('No summary table available yet. Run the evaluation cell in Section 8 first.')

paper_style_summary_df


Saved: /Users/juansantiago/Desktop/MIT Classes 2023-2027/Spring 2026/6.S955/FinalProject/6.S955_project/outputs/stage2_cnn_replication/results/paper_style_summary.csv


,Experiment,Split,Method,Accuracy Mean (%),Accuracy Std (%),Macro F1 Mean (%),Macro F1 Std (%)
0,Experiment 1,folds,CNN,85.925926,2.371527,84.923566,3.661660
1,Experiment 2,guitars,CNN,88.342441,3.037037,85.776427,3.655264
2,Experiment 3,amplifiers,CNN,91.621129,4.894364,91.215493,5.117885
